<a href="https://colab.research.google.com/github/Musamehar/ML_Intership/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1.Unit of Analysis (Grain): One row represents a single content item (content_id) for a specific client (client_id) evaluated at a single point in time (or month snapshot).

2.Tables Used: Primary starter dataset data/raw/content_refresh_anonymized.csv (mirroring dim_content + fact_content_daily_performance aggregated metrics).

3.Time Window: Trailing 90-day performance feature window evaluating upcoming performance decay.

4.Target / Proxy to Predict/Rank: decay_flag (is_declining_label / trend_direction == 'down').

5.Deliberately Excluded: trend_pct and trend_direction — because they directly derive the decay label and would cause circular target leakage.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [4]:
import os
import pandas as pd
from pathlib import Path

if not os.path.exists('ML_Intership') and not Path('data/raw/content_refresh_anonymized.csv').exists():
    !git clone https://github.com/Musamehar/ML_Intership.git

possible_paths = [
    Path('ML_Intership/data/raw/content_refresh_anonymized.csv'),
    Path('data/raw/content_refresh_anonymized.csv'),
    Path('../data/raw/content_refresh_anonymized.csv')
]

data_path = next((p for p in possible_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not locate data/raw/content_refresh_anonymized.csv.")

df = pd.read_csv(data_path)

# FACT 1: Grain Uniqueness Check
total_rows = len(df)
unique_content = df['content_id'].nunique()
print(f"FACT 1 - Grain Uniqueness: Total Rows = {total_rows:,} | Unique content_ids = {unique_content:,}")
assert total_rows == unique_content, "Grain mismatch detected!"

# FACT 2: Active Slice Survival Count
active_slice = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].drop_duplicates('content_id').copy()
print(f"FACT 2 - Qualified Slice Count: {len(active_slice):,} rows out of {total_rows:,} total rows survive filters.")

# FACT 3: Target Availability & Rate
if 'is_declining_label' in active_slice.columns:
    active_slice['decay_flag'] = active_slice['is_declining_label'].astype(int)
else:
    active_slice['decay_flag'] = (active_slice['trend_direction'] == 'down').astype(int)

print(f"FACT 3 - Target Availability: {active_slice['decay_flag'].sum():,} rows ({active_slice['decay_flag'].mean()*100:.1f}%) exhibit active decay.")

FACT 1 - Grain Uniqueness: Total Rows = 30,000 | Unique content_ids = 30,000
FACT 2 - Qualified Slice Count: 30,000 rows out of 30,000 total rows survive filters.
FACT 3 - Target Availability: 16,262 rows (54.2%) exhibit active decay.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
import numpy as np

# Construct a 5-feature dataframe from pre-decision, knowable signals:
feature_df = pd.DataFrame()
feature_df['content_id'] = active_slice['content_id']

# Feature 1: Log Impressions (Knowable from historical Search Console visibility)
feature_df['log_impressions_90d'] = np.log1p(active_slice['impressions_90d'])

# Feature 2: CTR 90d (Knowable from historical clicks / impressions ratio)
feature_df['ctr_90d'] = active_slice['clicks_90d'] / (active_slice['impressions_90d'] + 1e-5)

# Feature 3: Average SERP Position (Knowable historical rank)
feature_df['avg_position'] = active_slice['avg_position']

# Feature 4: Content Age Days (Knowable from content publishing metadata)
feature_df['content_age_days'] = active_slice['content_age_days']

# Feature 5: Keyword Presence Indicator (Knowable from setup metadata)
kw_col = next((col for col in ['primary_keyword', 'keyword_hash_id', 'main_keyword'] if col in active_slice.columns), None)
if kw_col:
    feature_df['has_keyword'] = active_slice[kw_col].notna().astype(int)
else:
    feature_df['has_keyword'] = (active_slice['word_count'] > 0).astype(int)

feature_df['target_decay'] = active_slice['decay_flag']

print("Five-Feature Frame Preview:")
print(feature_df.head())

Five-Feature Frame Preview:
             content_id  log_impressions_90d   ctr_90d  avg_position  \
0  content_304f48230142             8.243808  0.007626          10.6   
1  content_a1fb4e703a9e             9.636980  0.000457          20.3   
2  content_9aa793d4d895             9.440023  0.000874          36.5   
3  content_331d6c4de07b             9.371779  0.004936           6.2   
4  content_d99b7a2d90ca             9.859588  0.001254          44.0   

   content_age_days  has_keyword  target_decay  
0               187            1             1  
1               445            1             1  
2               141            1             1  
3               463            0             0  
4               263            1             1  


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 1. Evaluate Honest Features Baseline
X_honest = feature_df[['avg_position', 'content_age_days', 'log_impressions_90d', 'ctr_90d', 'has_keyword']]
y = feature_df['target_decay']

rf_honest = RandomForestClassifier(n_estimators=50, random_state=42)
rf_honest.fit(X_honest, y)
honest_score = roc_auc_score(y, rf_honest.predict_proba(X_honest)[:, 1])
print(f"1. Honest Feature Baseline ROC-AUC: {honest_score:.4f}")

# 2. Inject TRAP Feature ('trend_pct' - used to derive the target)
X_leaked = X_honest.copy()
X_leaked['leaked_trend_signal'] = active_slice['trend_pct']

rf_leaked = RandomForestClassifier(n_estimators=50, random_state=42)
rf_leaked.fit(X_leaked, y)
leaked_score = roc_auc_score(y, rf_leaked.predict_proba(X_leaked)[:, 1])
print(f"2. TRAP / LEAKED Feature ROC-AUC: {leaked_score:.4f} (Artificially inflated / Invalid!)")

# 3. Clean up the trap column
del X_leaked
print("\n[ACTION]: Deleted leaked column. Retaining honest score for modeling.")

1. Honest Feature Baseline ROC-AUC: 0.9999
2. TRAP / LEAKED Feature ROC-AUC: 1.0000 (Artificially inflated / Invalid!)

[ACTION]: Deleted leaked column. Retaining honest score for modeling.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.